In [1]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')


/home/sanjay/Projects/Learning/RAG/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3152.07it/s]


In [5]:
import fitz
import os

def extract_text(folderPath):
    documents = []

    for file in os.listdir(folderPath):
        if file.endswith("pdf"):
            path = os.path.join(folderPath, file)

            pdf = fitz.open(path)
            text = ""

            for page in pdf:
                text += page.get_text()
            
            documents.append({
                "source": file,
                "content": text
            })
    return documents

def chunk_text(text, chunk_size=1000, overlap=500):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size 
        chunks.append(text[start:end])
        start += chunk_size-overlap
    
    return chunks
docs = extract_text("Data")

allChunks = []
for doc in docs:
    chunks = chunk_text(doc["content"])
    for chunk in chunks:
        allChunks.append({
            "source": doc['source'],
            "text": chunk
        })

print(len(chunks))


1108


In [6]:
text = [chunk['text'] for chunk in allChunks]

emb = model.encode(
    text,
    normalize_embeddings=True,
    convert_to_numpy=True
)

print(emb.shape)

(1108, 384)


In [7]:
import faiss
import pickle as pkl

dim = emb.shape[1]
index = faiss.IndexFlatIP(dim)

index.add(emb)

faiss.write_index(index, "faissIndex/index.bin")
with open("faissIndex/metadata.pkl", "wb") as f:
    pkl.dump(allChunks, f)



In [8]:
index = faiss.read_index("faissIndex/index.bin")

with open("faissIndex/metadata.pkl", "rb") as f:
    metadata = pkl.load(f)

print(metadata[0])

{'source': 'ubuntu.pdf', 'text': 'Praise for Previous Editions of \nThe Official Ubuntu Book\n“The Ofﬁcial Ubuntu Book is a great way to get you started with Ubuntu,\ngiving you enough information to be productive without overloading you.”\n—John Stevenson, DZone book reviewer\n“OUB is one of the best books I’ve seen for beginners.”\n—Bill Blinn, TechByter Worldwide\n“This book is the perfect companion for users new to Linux and Ubuntu.\nIt covers the basics in a concise and well-organized manner. General use\nis covered separately from troubleshooting and error-handling, making\nthe book well-suited both for the beginner as well as the user that needs\nextended help.”\n—Thomas Petrucha, Austria Ubuntu User Group\n“I have recommended this book to several users who I instruct regularly on\nthe use of Ubuntu. All of them have been satisﬁed with their purchase and\nhave even been able to use it to help them in their journey along the way.”\n—Chris Crisafulli, Ubuntu LoCo Council, \nFlorid

## Evaluation Pipeline 

Generate a "golden" dataset generated by randomly selecting chunks and providing it to a llm and generate 5 questions


In [9]:
print(len(metadata))

1108


In [10]:
import random
selectedChunks = []
for _ in range(5):
    selectedChunks.append(random.randint(40, len(chunks)))
                          
selectedChunks

[591, 459, 63, 853, 197]

In [16]:
prompt = """You are an helpful assistant for me to learn RAG
I am currently learning how to evaluate RAG, for that I need a golden dataset
So given the provided context, Strictly Generate 3 meaningful questions ONLY from that context
No extra words should be produced. And do not provide questions like 'What is the summary of the provided context'
since we are evaluating the retriever, we don't need that kind of questions to evaluate the retrieval chunks

Context: {chunk}
Questions: 
"""

In [17]:
from groq import Groq

client = Groq(api_key=os.environ["GROQ_API_KEY"])
res = {}
for c in selectedChunks:
    chunk = metadata[c]
    response = client.chat.completions.create(
        model = "llama-3.3-70b-versatile",
        messages=[{
            "role": "user",
            "content": prompt.format(chunk=chunk)
        }],
        temperature=0,
        max_tokens = 512
    )

    answer = response.choices[0].message.content
    answer = answer.split('\n')
    for i in answer:
        res[i] = c

In [18]:
for k, v in res.items():
    print(k, v)

1. How to customize Ubuntu for performance and accessibility? 591
2. What are the basic commands for navigating the Ubuntu filesystem? 591
3. How to get help on the command line in Ubuntu? 591
1. How does the window behave when the minimize key combo is pressed if it's already minimized? 459
2. What is the philosophy behind UNIX in terms of creating tools? 459
3. What is the primary function of the ls command in the command-line environment? 459
1. Who is the president of the North Bay Linux Users' Group? 63
2. What is the title of the book written by Jono Bacon? 63
3. Where is Benjamin Mako Hill currently a fellow? 63
1. What is the primary function of Launchpad in relation to Ubuntu development? 853
2. What types of tracking software does Launchpad provide to free software projects? 853
3. Is Launchpad exclusively used for Ubuntu development or can it be used by other projects? 853
1. What type of backup can Canonical offer to organizations? 197
2. What is the role of Bazaar in the d

In [20]:
k = 3
TP = 0
for q, cId in res.items():
    qEmb = model.encode([q], normalize_embeddings=True, convert_to_numpy=True)
    score, indices = index.search(qEmb, k)
    print(q, cId, indices)
    if cId in indices:
        TP+=1

    
recall = TP/len(res) * 100
print(recall)
    

1. How to customize Ubuntu for performance and accessibility? 591 [[590 563 290]]
2. What are the basic commands for navigating the Ubuntu filesystem? 591 [[609 610 439]]
3. How to get help on the command line in Ubuntu? 591 [[591 640 639]]
1. How does the window behave when the minimize key combo is pressed if it's already minimized? 459 [[458 457 456]]
2. What is the philosophy behind UNIX in terms of creating tools? 459 [[460 128  44]]
3. What is the primary function of the ls command in the command-line environment? 459 [[595 460 596]]
1. Who is the president of the North Bay Linux Users' Group? 63 [[63 62 28]]
2. What is the title of the book written by Jono Bacon? 63 [[58 56 57]]
3. Where is Benjamin Mako Hill currently a fellow? 63 [[ 64  63 189]]
1. What is the primary function of Launchpad in relation to Ubuntu development? 853 [[202 853 201]]
2. What types of tracking software does Launchpad provide to free software projects? 853 [[853 857 855]]
3. Is Launchpad exclusively us

## Analysis

- Increasing the chunk size from 500 - 1000 increased recall from 53.65 - 66.67
- spliting chunks per sentence is costly, more than 50000 chunks are retrieved and computation is high
- Splitting chunks per paragraph, it still provided 11000 chunks and recall is 46.15 too low (it captured unnecassary texts like in the first 10 pages of the book)